# AlgoTrade — training and validation

Fits `ManipulationModel` and asks the four questions that decide whether the
strategy is tradable at all:

1. **Does the predicted sigma reach the entry gate?** The gate is an *absolute*
   sigma (`h5 >= 3.0 AND h10 >= 2.0`), so a fit whose output is narrower than
   bist's never fires. This is the first thing to check after any retrain.
2. **Does the ranking hold out of sample?** The model is a ranker; bist's own
   caveat is that the sigma is not a calibrated forecast, so decile lift is the
   honest read rather than an R².
3. **What does the gate actually select?** Count, names, dates — and the overlap
   with limit-up closes, which is the caveat that governs how to read any P&L.
4. **What is it splitting on?** Gain-based feature importance per head.

## Prerequisites

The venv needs a kernel, and the optional plots need matplotlib:

```sh
app/python/.venv/bin/pip install ipykernel matplotlib
app/python/.venv/bin/python -m ipykernel install --user --name stonks-venv \
    --display-name "stonks (app/python/.venv)"
```

Then pick **stonks (app/python/.venv)** as the kernel. It must be the same
interpreter the build links against — see `CLAUDE.md` → macOS presets. Every
table below works without matplotlib; only the final cell needs it.

Run order is top to bottom. The training cell takes about a minute and the
scoring cell somewhat longer, since both build 72 features over the whole panel.

## Setup

The notebook lives in `tools/` but every path in this project is repo-root
relative — `AlgoTradeStrategy.artifact` included — so this walks up to the root
and works from there.

In [ ]:
import logging
import os
import pathlib
import sys

import numpy as np
import pandas as pd

ROOT = pathlib.Path.cwd()
while not (ROOT / "app" / "python" / "algo_trade.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "app" / "python"))

import algo_trade
from algo_trade import (
    ENTRY_H10_MIN,
    ENTRY_H5_MIN,
    FEATURE_COVERAGE,
    FEATURE_NAMES,
    LIMIT_HIT,
    MIN_HISTORY,
    ManipulationModel,
    _design_matrix,
    _features,
    _labels,
    _panel,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print("repo root:", ROOT)

## Configuration

`TRAIN_START` defaults to bist's **backtest** window (full history), not its
screen's 2024-01-01. That pairing is load-bearing: the entry gate is an absolute
sigma, and a fit restricted to 2024 predicts far too tightly for `h5 >= 3.0` to
fire. See `BACKTEST_TRAIN_START` in `algo_trade.py`.

In [ ]:
DATA = "app/data/bist_1d.parquet"
TRAIN_END = "2024-12-31"                        # last bar the fit may see
TRAIN_START = algo_trade.BACKTEST_TRAIN_START    # "2000-01-01" — full history

# True overwrites the artifact the strategy loads at run time. Left off so the
# notebook is safe to re-run while a backtest is being read.
SAVE_ARTIFACT = False
ARTIFACT = algo_trade.AlgoTradeStrategy.artifact

## Load the feed

`_panel` applies bist's row drops (halts, zero-volume-with-price, disordered
OHLC) and pivots to `{field: (sessions, symbols)}`. Missing cells are NaN, which
every feature already expects.

In [ ]:
frame = pd.read_parquet(DATA)
panel = _panel(frame)
index = panel["close"].index
cutoff = algo_trade._as_index_value(TRAIN_END, index)
oos = index.to_numpy() > cutoff

print(f"{len(frame):,} rows, {frame['symbol'].nunique()} symbols")
print(f"panel: {len(index)} sessions x {panel['close'].shape[1]} symbols, "
      f"{pd.Timestamp(index[0]).date()} -> {pd.Timestamp(index[-1]).date()}")
print(f"in-sample: {int((~oos).sum())} sessions through {TRAIN_END}")
print(f"out-of-sample: {int(oos.sum())} sessions")

## Train

`train` truncates to `TRAIN_END` *before* building labels, so a label whose
forward window would reach past the cutoff resolves to NaN and is dropped. The
embargo is structural, not a convention to remember.

The per-head log line is the first read on whether this fit is usable: watch
`pred q999` against the 3.0 / 2.0 entry gates.

In [ ]:
%%time
model = ManipulationModel(train_start=TRAIN_START).train(panel, train_end=TRAIN_END)

In [ ]:
if SAVE_ARTIFACT:
    model.save(ARTIFACT)
    print("wrote", ARTIFACT)
else:
    print(f"not saved — set SAVE_ARTIFACT = True to overwrite {ARTIFACT}")
    print("the strategy keeps loading whatever is already on disk")

## Score the out-of-sample stretch

Three things here have to match what the strategy does, or the validation
measures a different model than the one that trades:

- **Features come from the whole panel**, so `obv` and `days_since_past_extreme`
  accumulate from each symbol's first bar. Scoring a 300-bar window instead
  rebases both — that is exactly what `ScoringState` exists to repair.
- **Winsorize bounds are the last fitted date's**, which is what `signal`
  forward-fills onto every date past the training window.
- **The row filters are `signal`'s**: enough finite features, enough listed
  history, and inside the artifact's liquid universe.

Labels come from the full panel so forward windows resolve. The final `horizon`
sessions cannot resolve and stay NaN by construction.

In [ ]:
%%time
F = _features(panel)

# Row filters, applied before F is released.
listed = np.isfinite(panel["close"].to_numpy(dtype=float))
age = np.cumsum(listed, axis=0)
required = max(1, int(FEATURE_COVERAGE * len(FEATURE_NAMES)))
usable = (np.isfinite(F).sum(axis=2) >= required) & (age >= MIN_HISTORY)
if model.liquid is not None:
    usable &= np.array([s in model.liquid for s in panel["close"].columns])[None, :]

lo, hi = model._bounds
X = _design_matrix(np.clip(F, lo, hi))
del F

mask = usable & oos[:, None]
print(f"{mask.sum():,} scoreable out-of-sample rows "
      f"({int(oos.sum())} sessions, {usable[oos].any(axis=0).sum()} distinct symbols)")

In [ ]:
symbols = panel["close"].columns.to_numpy()
di, si = np.nonzero(mask)
ret = (panel["close"] / panel["close"].shift(1) - 1.0).to_numpy()

scores = pd.DataFrame({"date": index.to_numpy()[di], "symbol": symbols[si],
                       "ret": ret[mask]})
for head in model.heads:
    scores[head.name] = model._predict(head, X[mask])
    scores[f"{head.name}_y"] = _labels(panel, head)[mask]

print(f"{len(scores):,} rows | "
      f"{scores['up_h5_y'].notna().sum():,} with a resolved up_h5 label")
scores.head()

### 1. Does the predicted sigma reach the entry gate?

The single most decision-relevant table. The gate is absolute, so if `q99.9`
sits below 3.0 the strategy will take approximately no trades no matter how good
the ranking is. A fit restricted to 2024 reaches 3.088 at q99.9 and produces two
trades in eighteen months; full history widens the output, which is what makes
bist's thresholds mean what bist intended.

In [ ]:
q = [0.5, 0.9, 0.99, 0.999, 0.9999, 1.0]
scale = pd.DataFrame({h.name: scores[h.name].quantile(q) for h in model.heads})
scale.index = [f"q{100 * x:g}" for x in q]
display(scale.round(3))

print("rows clearing each gate on its own:")
for name, floor in (("up_h5", ENTRY_H5_MIN), ("up_h10", ENTRY_H10_MIN)):
    hit = scores[name] >= floor
    print(f"  {name:>7} >= {floor}: {int(hit.sum()):>6,} rows  ({hit.mean():.4%})")

### 2. Does the ranking hold out of sample?

Decile lift: bucket predictions into ten, then read the realized label per
bucket. What matters is that the top decile beats the bottom and that the
relationship is monotone-ish — not that predictions equal realizations, which
bist explicitly disclaims.

`hit_rate` is the share of rows whose realized label cleared `k_sigma`, the
model's own definition of an extreme move. Remember the up labels are
drawdown-gated: a move that first dipped past -10% is recorded as zero, so this
measures *tradable* moves rather than moves.

In [ ]:
for head in model.heads:
    d = scores[[head.name, f"{head.name}_y"]].dropna()
    if d.empty:
        print(f"{head.name}: no resolved labels out of sample")
        continue
    rho = d[head.name].corr(d[f"{head.name}_y"], method="spearman")
    bucket = pd.qcut(d[head.name], 10, labels=False, duplicates="drop")
    table = d.groupby(bucket).agg(
        n=(head.name, "size"),
        pred_mean=(head.name, "mean"),
        realized_mean=(f"{head.name}_y", "mean"),
        realized_median=(f"{head.name}_y", "median"),
        hit_rate=(f"{head.name}_y", lambda s: (s >= model.k_sigma).mean()),
    )
    print(f"\n=== {head.name} — {len(d):,} resolved rows | "
          f"Spearman {rho:+.4f} ===")
    display(table.round(4))

### 3. What does the strategy's gate actually select?

The conjunction the strategy trades, plus the caveat that governs how to read
any P&L that comes out of it: on the artifact as fitted, **every** signal in the
out-of-sample stretch was a name that closed at the +10% price band. The
strategy buys the next open, so those are fills a real order book would not have
given. `skip_limit_locked = 1` removes them — and historically leaves nothing.

In [ ]:
gate = scores[(scores["up_h5"] >= ENTRY_H5_MIN)
              & (scores["up_h10"] >= ENTRY_H10_MIN)].copy()
print(f"{len(gate)} bars clear h5 >= {ENTRY_H5_MIN} AND h10 >= {ENTRY_H10_MIN}")

if len(gate):
    gate["limit_up"] = gate["ret"] >= LIMIT_HIT
    gate["composite"] = (gate["up_h5"] + gate["up_h10"]) / 2.0
    print(f"of those, {int(gate['limit_up'].sum())} closed at the "
          f"+{100 * LIMIT_HIT:.0f}% band ({gate['limit_up'].mean():.0%}) — "
          f"unbuyable at the next open in practice")
    print(f"{gate['symbol'].nunique()} distinct symbols, "
          f"{pd.Timestamp(gate['date'].min()).date()} -> "
          f"{pd.Timestamp(gate['date'].max()).date()}")
    display(gate[["date", "symbol", "up_h5", "up_h10", "dn_h5", "composite",
                  "up_h5_y", "up_h10_y", "ret", "limit_up"]]
            .sort_values("date").round(3).reset_index(drop=True))
else:
    print("nothing fires — check the q99.9 row in section 1 before anything else")

### 4. What is it splitting on?

Gain-based importance, normalised to each head's share of total gain. A feature
absent from a booster never got split on and reads 0.

In [ ]:
imp = pd.DataFrame({h.name: pd.Series(
    model.models[h.name].get_score(importance_type="gain")) for h in model.heads})
imp = imp.reindex(FEATURE_NAMES).fillna(0.0)
imp = imp.div(imp.sum(axis=0), axis=1)
imp["mean"] = imp.mean(axis=1)
display(imp.sort_values("mean", ascending=False).head(20).round(4))

### Plots (optional)

Needs matplotlib. Everything above is complete without it.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not in the venv — the tables above are the whole story.")
    print("  app/python/.venv/bin/pip install matplotlib")
else:
    gates = {"up_h5": ENTRY_H5_MIN, "up_h10": ENTRY_H10_MIN}
    fig, axes = plt.subplots(1, len(model.heads),
                             figsize=(4.5 * len(model.heads), 3.4))
    for ax, head in zip(np.atleast_1d(axes), model.heads):
        ax.hist(scores[head.name].dropna(), bins=80)
        ax.set_yscale("log")
        ax.set_title(f"{head.name} — predicted sigma")
        if head.name in gates:
            ax.axvline(gates[head.name], color="crimson", lw=1.2)
            ax.set_xlabel(f"entry gate at {gates[head.name]}")
    fig.tight_layout()
    plt.show()